# raw_to_dst — Convert raw manufacturer files to standard tag folder

Run this **once per tag** before launching the main pipeline.

---

## What this notebook does

It reads the raw files from the tag manufacturer (Wildlife Computers or Lotek) and writes a
**standard tag folder** that the pipeline can open with `load_tag()`:

```
{TAG_ROOT}/{tag_name}/
    dst.csv            ← time, temperature, pressure, light
    tagging_event.csv  ← release + recapture/death events
    metadata.json      ← tag_name, tag_type
    lightloc.csv       ← (optional) twilight crossing times for PSAT-light tags
```

---

## Tag types and source files

Wildlife Computers MiniPAT tags can produce different file sets depending on whether the tag
was physically recovered or only transmitted data via Argos satellite.

### `"lotek"` — Lotek LAT2810

Raw semicolon-separated CSV at 5-second resolution.  Contains raw light counts
(`LightIntensity`), external temperature, and pressure.

**Source file:** `*.csv` (semicolon-separated, comma as decimal separator)
**Light in `dst.csv`:** ✅ filled (`light` column = raw counts)
**In the pipeline:** `HAS_LIGHT = True`, `LIGHT_SOURCE = "series"`

---

### `"wc_archival"` — Wildlife Computers MiniPAT, **physically recovered**

The tag was recovered and plugged into the WC DAP software, which produced an
`out-Archive.csv` file containing the full high-resolution archive (typically 15-second
records) with a `Light Level` column.  This is the richest data source.

**Source file:** `out-Archive.csv`
**Light in `dst.csv`:** ✅ filled (`light` column = raw counts from Archive)
**In the pipeline:** `HAS_LIGHT = True`, `LIGHT_SOURCE = "series"`

> Also produced by DAP: `out-LightLoc.csv` with `Source = DAP` — these are crossing times
> derived from the Archive by WC software.  They are **not used** here: we prefer to
> re-detect crossings ourselves from the raw `out-Archive.csv` series (better control over
> threshold and calibration).

---

### `"wc_psat"` — Wildlife Computers MiniPAT, **not recovered, no usable light**

The tag was never physically recovered.  It popped up and transmitted a summary via Argos,
but the transmission did not include light crossing times (or the transmission was too
incomplete).  Only depth and temperature data are available at 10-minute resolution via
the `*-Series.csv` file.

**Source file:** `{serial}-Series.csv`
**Light in `dst.csv`:** ❌ `light = NaN`
**In the pipeline:** `HAS_LIGHT = False` (solar and lunar chapters are skipped entirely)

---

### `"wc_psat_light"` — Wildlife Computers MiniPAT, **not recovered, but transmitted twilight crossings**

The tag was not physically recovered, but it successfully transmitted twilight crossing
times via Argos before or at pop-up.  Wildlife Computers encodes these as a
`*-LightLoc.csv` file with `Source = Transmission`.

Each row is one twilight event (Dawn or Dusk) with:
- `Time` — the crossing time as computed by the WC on-board algorithm (threshold method)
- `Type` — `Dawn` (sunrise) or `Dusk` (sunset)
- `MaxDepth` — maximum depth during the twilight window (dbar), used as a quality filter
- `LL0`…`LLN` — raw light samples around the crossing (N varies by firmware, typically 12–17)
- `Delta` — time interval between samples (seconds, variable: 300–1800 s)

Note: this is the **same threshold method** as for archival tags, just computed on-board
by WC instead of by our pipeline.  The crossing time precision is comparable, but the
threshold is not recalibrated to the individual fish — it is fixed at deployment.
This is why `THRESH_DEG_INIT` (rather than a self-calibrated value) is used in the pipeline
when `LIGHT_SOURCE = "lightloc"`.

Coverage is much lower than archival: Argos bandwidth limits how many twilights can be
transmitted.  Typically only 5–10 % of nights make it through (e.g. 46 out of 895 for a
15-month deployment).

**Source files:** `{serial}-Series.csv` (temperature + depth) + `{serial}-LightLoc.csv`
**Light in `dst.csv`:** ❌ `light = NaN` (no raw series)
**Extra file:** ✅ `lightloc.csv` written to the tag folder
**In the pipeline:** `HAS_LIGHT = True`, `LIGHT_SOURCE = "lightloc"`

---

## Summary table

| `TAG_TYPE` | Source file(s) | `light` in dst.csv | `lightloc.csv` | Pipeline setting |
|---|---|---|---|---|
| `"lotek"` | `*.csv` (Lotek) | ✅ raw counts | — | `HAS_LIGHT=True`, `LIGHT_SOURCE="series"` |
| `"wc_archival"` | `out-Archive.csv` | ✅ raw counts | — | `HAS_LIGHT=True`, `LIGHT_SOURCE="series"` |
| `"wc_psat"` | `*-Series.csv` | ❌ NaN | — | `HAS_LIGHT=False` |
| `"wc_psat_light"` | `*-Series.csv` + `*-LightLoc.csv` | ❌ NaN | ✅ | `HAS_LIGHT=True`, `LIGHT_SOURCE="lightloc"` |


In [ ]:
import sys
sys.path.append("../")

TAG_ROOT = "/tmp/tags"  # local output directory

TAG_TYPE = "wc_psat"
tag_name = "263936"
RAW_DATA_PATH = (
    "../docs/specs/reference/tuna-tristan/tuna-tristan/263936/263936-Series.csv"
)
TAGGING_EVENTS_PATH = "263936/tagging_events.csv"

# ── S3 upload (optional) ─────────────────────────────────────────────────────
SAVE_TO_S3 = False  # set True to upload the tag folder to S3 after conversion

scratch_root = "s3://gfts-ifremer/tuna_mediterranean/tags/formatted"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}


In [ ]:
from pangeo_fish.light.ingest import prepare_tag_folder

prepare_tag_folder(
    RAW_DATA_PATH,
    TAG_TYPE,
    tagging_events_path=TAGGING_EVENTS_PATH,
    output_dir=TAG_ROOT,
    tag_name=tag_name,
)

## Verify

In [ ]:
import sys

sys.path.append("../")
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(tag_root=TAG_ROOT, tag_name=tag_name)
print(f"Deployment: {time_slice.start} → {time_slice.stop}")
print(f"tag_log: {len(tag_log.time):,} timesteps")
tag

In [ ]:
if SAVE_TO_S3:
    import s3fs

    s3 = s3fs.S3FileSystem(**storage_options)
    local_folder = f"{TAG_ROOT}/{tag_name}"
    s3_folder = f"{scratch_root}/{tag_name}"

    files = ["dst.csv", "tagging_event.csv", "metadata.json"]
    if TAG_TYPE == "wc_psat_light":
        files.append("lightloc.csv")

    for fname in files:
        s3.put(f"{local_folder}/{fname}", f"{s3_folder}/{fname}")
        print(f"Uploaded → {s3_folder}/{fname}")

    print(f"\nLoad with:")
    print(f'  load_tag(tag_root="{scratch_root}", tag_name="{tag_name}")')
else:
    print("SAVE_TO_S3=False — files kept locally in:")
    print(f"  {TAG_ROOT}/{tag_name}/")
    print(f"\nLoad with:")
    print(f'  load_tag(tag_root="{TAG_ROOT}", tag_name="{tag_name}")')
